# IEP-1002 Day 2 — 구조 기반 청킹


## Step 1 — 헤딩 전처리

Day 1에서 저장한 `ipcc_headings.json` (150개)을 입력값으로 받아 세 가지 정제 작업을 수행합니다.

1. **목차 중복 제거** — p.13~14는 목차 페이지로, 본문과 동일한 헤딩이 반복 등장합니다. 이 페이지의 헤딩은 청크 경계로 사용하지 않습니다.
2. **러닝헤더 제거** — `섹션 2`, `섹션 3` 처럼 제목 없이 섹션 번호만 있는 항목은 페이지 상단의 러닝헤더입니다.
3. **수동 오탐 제거** — Day 1 육안 확인에서 발견된 잔여 오탐 3개를 제거합니다.

완료 기준: 중복 헤딩 0건, `ipcc_headings_clean.json` 저장


### 셀 1 — 데이터 로드 및 현황 파악
- 확인할 것: 150개 헤딩이 정상 로드되는지, 패턴별 분포가 Day 1 결과와 일치하는지.


In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

OUTPUT_DIR = "/content/drive/MyDrive/IPCC_data/parsed"

# Day 1 산출물 로드
headings_path = f"{OUTPUT_DIR}/ipcc_headings.json"
with open(headings_path, encoding="utf-8") as f:
    headings_raw = json.load(f)

print(f"로드 완료: {len(headings_raw)}개")
print()

# 패턴별 분포 확인 — Day 1 결과와 비교
pattern_counts = Counter(h["pattern"] for h in headings_raw)
print(f"{'패턴명':20s}  {'수':>4s}  {'Day 1 기준':>8s}  {'일치':>4s}")
print("─" * 46)
DAY1_EXPECTED = {
    "알파벳점": 8, "숫자점숫자": 32, "숫자점숫자점숫자": 28,
    "CrossSection": 8, "섹션번호": 46, "Box": 3,
    "숫자점": 2, "부속서": 23,
}
for pattern, cnt in sorted(pattern_counts.items(), key=lambda x: -x[1]):
    expected = DAY1_EXPECTED.get(pattern, "?")
    ok = "✓" if cnt == expected else "✗"
    print(f"  {pattern:18s}  {cnt:4d}  {str(expected):>8s}  {ok:>4s}")
print(f"  {'합계':18s}  {len(headings_raw):4d}  {'150':>8s}  {'✓' if len(headings_raw)==150 else '✗':>4s}")
print()

# 페이지 범위 확인
pages = [h["page"] for h in headings_raw]
print(f"페이지 범위: p.{min(pages)} ~ p.{max(pages)}")
print(f"목차 구간(p.13~14) 헤딩 수: {sum(1 for h in headings_raw if h['page'] in (13, 14))}개  ← Step 2에서 제거")


### 셀 2 — 목차 중복 헤딩 제거 (p.13~14)
- 확인할 것: 제거 후 동일 헤딩 텍스트가 본문 페이지에 정상적으로 남아 있는지.


In [ ]:
# ── 필터 1: 목차 페이지 헤딩 제거 ─────────────────────────
# p.13~14는 목차 페이지. 본문과 동일 헤딩이 중복 등장 → 청크 경계로 사용 불가
TOC_PAGES = {13, 14}

removed_toc   = [h for h in headings_raw if h["page"] in TOC_PAGES]
after_toc     = [h for h in headings_raw if h["page"] not in TOC_PAGES]

print(f"목차 페이지 헤딩 제거: {len(removed_toc)}개")
print(f"  제거 전: {len(headings_raw)}개")
print(f"  제거 후: {len(after_toc)}개")
print()

# 제거된 헤딩 목록 출력
print("── 제거된 목차 헤딩 ─────────────────────────────────")
for h in removed_toc:
    print(f"  p.{h['page']:3d}  [{h['pattern']:18s}]  {h['text'][:60]}")
print()

# 교차 검증: 제거된 헤딩 텍스트가 본문에 동일하게 남아있는지 확인
toc_texts = {h["text"] for h in removed_toc}
body_texts = {h["text"] for h in after_toc}
orphaned   = toc_texts - body_texts  # 목차에만 있고 본문에 없는 것 (비정상)

print(f"교차 검증 — 목차 헤딩이 본문에도 존재하는지:")
if orphaned:
    print(f"  ⚠ 본문에 없는 헤딩 {len(orphaned)}개 발견:")
    for t in sorted(orphaned):
        print(f"    · {t[:70]}")
    print("  → 해당 헤딩은 목차 전용이므로 제거해도 무방합니다.")
else:
    print(f"  ✓ 모든 목차 헤딩({len(toc_texts)}개)이 본문에 정상 존재 — 안전하게 제거됨")


### 셀 3 — 러닝헤더 제거 (섹션번호 패턴)
- 확인할 것: `섹션 2: 현황 및 추세` 처럼 콜론+제목이 있는 것은 **남기고**, `섹션 2` 처럼 번호만 있는 것만 제거하는지.


In [ ]:
# ── 필터 2: 러닝헤더 제거 ──────────────────────────────────
# 패턴: 섹션번호 (^섹션\s*\d+)
# 제목 있는 것: "섹션 2: 현황 및 추세"  → 실제 경계, 유지
# 제목 없는 것: "섹션 2"               → 러닝헤더, 제거

def is_running_header(h: dict) -> bool:
    """True → 러닝헤더 (제거 대상)"""
    if h["pattern"] != "섹션번호":
        return False
    text = h["text"].strip()
    # 콜론(:) 뒤에 실제 제목이 있으면 진짜 헤딩
    if re.search(r"섹션\s*\d+\s*[:：]\s*\S", text):
        return False
    # 숫자 뒤 바로 끝나거나 공백만 있으면 러닝헤더
    return bool(re.match(r"^섹션\s*\d+\s*$", text))

removed_running = [h for h in after_toc if is_running_header(h)]
kept_section    = [h for h in after_toc if h["pattern"] == "섹션번호" and not is_running_header(h)]
after_running   = [h for h in after_toc if not is_running_header(h)]

print(f"러닝헤더 제거: {len(removed_running)}개")
print(f"  섹션번호 전체:  {sum(1 for h in after_toc if h['pattern']=='섹션번호')}개")
print(f"  러닝헤더 제거:  {len(removed_running)}개")
print(f"  실제 헤딩 유지: {len(kept_section)}개")
print()

print("── 제거된 러닝헤더 ─────────────────────────────────")
for h in removed_running:
    print(f"  p.{h['page']:3d}  {h['text'][:60]}")
print()

print("── 유지된 섹션 헤딩 (실제 경계) ────────────────────")
for h in kept_section:
    print(f"  p.{h['page']:3d}  {h['text'][:70]}")
print()
print(f"제거 후 헤딩 수: {len(after_running)}개")


### 셀 4 — 수동 오탐 제거 (잔여 3개)
- 확인할 것: 출력된 후보 목록을 육안으로 보고, 오탐이 맞으면 `MANUAL_REMOVE_PAGES` 세트에 페이지 번호를 확정합니다.
- Day 1 메모: p.33 그림 캡션 / p.99 그림 범례 / p.100 본문 참조


In [ ]:
# ── 필터 3: 수동 오탐 제거 ─────────────────────────────────
# Day 1에서 육안 확인 시 발견한 잔여 오탐 3개
# 아래 셀을 실행하면 해당 페이지의 헤딩 후보가 출력됩니다.
# 출력 결과를 보고 오탐이 맞으면 MANUAL_REMOVE에 추가하세요.

SUSPECT_PAGES = [33, 99, 100]  # Day 1 메모 기반 후보 페이지

print("── 수동 확인 대상 헤딩 ──────────────────────────────")
for page in SUSPECT_PAGES:
    hits = [h for h in after_running if h["page"] == page]
    if hits:
        for h in hits:
            print(f"  [p.{page:3d}]  패턴: {h['pattern']:18s}  텍스트: {h['text'][:70]}")
    else:
        print(f"  [p.{page:3d}]  → 해당 페이지에 헤딩 없음 (이미 이전 필터에서 제거됨)")
print()

# ▼▼▼ 출력 결과를 보고 오탐을 직접 지정하세요 ▼▼▼
# 형식: (page번호, 텍스트 일부)  — 텍스트는 startswith 매칭에 사용
# 오탐이 없으면 빈 리스트로 두면 됩니다.
MANUAL_REMOVE = [
    # 예시: (33, "그림 2.1"),   # p.33 그림 캡션
    # 예시: (99, "범례"),        # p.99 그림 범례
    # 예시: (100, "Box.2,"),    # p.100 본문 참조
    # → 위 셀 출력 결과를 보고 실제 텍스트로 교체하세요
]

def is_manual_falsepositive(h: dict) -> bool:
    for page, text_prefix in MANUAL_REMOVE:
        if h["page"] == page and h["text"].startswith(text_prefix):
            return True
    return False

removed_manual = [h for h in after_running if is_manual_falsepositive(h)]
after_manual   = [h for h in after_running if not is_manual_falsepositive(h)]

print(f"수동 제거: {len(removed_manual)}개")
for h in removed_manual:
    print(f"  p.{h['page']:3d}  {h['text'][:70]}")
if not removed_manual:
    print("  (지정된 오탐 없음 — MANUAL_REMOVE가 비어 있거나 해당 항목이 이미 제거됨)")
print()
print(f"수동 제거 후 헤딩 수: {len(after_manual)}개")


### 셀 5 — 중복 헤딩 최종 점검
- 확인할 것: 동일한 (page, text) 쌍이 0건이어야 합니다. 있다면 원인을 파악하고 제거합니다.


In [ ]:
# ── 중복 헤딩 최종 점검 ────────────────────────────────────
# (page, text) 기준 중복 확인
seen    = set()
dupes   = []
deduped = []

for h in after_manual:
    key = (h["page"], h["text"])
    if key in seen:
        dupes.append(h)
    else:
        seen.add(key)
        deduped.append(h)

print(f"중복 점검 결과: {len(dupes)}건")
if dupes:
    print("  ⚠ 중복 헤딩 발견:")
    for h in dupes:
        print(f"    p.{h['page']:3d}  [{h['pattern']}]  {h['text'][:60]}")
    print(f"  → 중복 제거 후: {len(deduped)}개")
else:
    print("  ✓ 중복 없음")
    deduped = after_manual  # 변경 없음
print()

# 페이지 순서 보장 (청킹 로직이 순서에 의존)
deduped.sort(key=lambda h: (h["page"], h["text"]))
print("페이지 순 정렬 완료")


### 셀 6 — 전처리 결과 전체 출력 (육안 확인)
- 확인할 것: 청킹에 사용할 헤딩만 남았는지, 이상한 항목이 없는지 전체를 훑어보세요.


In [ ]:
# ── 전처리 결과 전체 출력 ──────────────────────────────────
# 계층 들여쓰기로 구조가 자연스러운지 확인
INDENT = {
    "섹션번호":          "",
    "알파벳점":          "  ",
    "숫자점":            "  ",
    "Box":              "  ",
    "CrossSection":     "  ",
    "부속서":            "",
    "숫자점숫자":         "    ",
    "숫자점숫자점숫자":    "      ",
}

print(f"{'페이지':>5s}  {'패턴':18s}  헤딩 텍스트")
print("─" * 82)
for h in deduped:
    indent = INDENT.get(h["pattern"], "  ")
    print(f"  p.{h['page']:3d}  {h['pattern']:18s}  {indent}{h['text'][:58]}")
print()
print(f"전체 {len(deduped)}개 헤딩 출력 완료")


### 셀 7 — 최종 저장 및 Step 1 완료 선언


In [ ]:
# ── 최종 저장 ──────────────────────────────────────────────
# Day 1 원본(ipcc_headings.json)은 보존하고, 전처리 결과는 별도 파일로 저장
clean_path = f"{OUTPUT_DIR}/ipcc_headings_clean.json"
with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(deduped, f, ensure_ascii=False, indent=2)

# 패턴별 최종 분포
final_counts = Counter(h["pattern"] for h in deduped)

print(f"""
╔══════════════════════════════════════════════════╗
  Day 2 — Step 1 헤딩 전처리 결과 요약
╚══════════════════════════════════════════════════╝

  Day 1 원본    : {len(headings_raw)}개
  목차 제거     : {len(removed_toc)}개  (p.13~14)
  러닝헤더 제거  : {len(removed_running)}개
  수동 오탐 제거 : {len(removed_manual)}개
  중복 제거     : {len(dupes)}건
  ─────────────────────────────
  최종 헤딩 수  : {len(deduped)}개
""")

print("  패턴별 최종 분포:")
for pattern, cnt in sorted(final_counts.items(), key=lambda x: -x[1]):
    bar = "█" * cnt
    print(f"    {pattern:20s}  {cnt:3d}개  {bar}")

print(f"""
  저장 파일:
    ✓ ipcc_headings_clean.json  ({len(deduped)}개, Day 2 Step 2 입력값)
    ✓ ipcc_headings.json        (Day 1 원본 보존)

  다음 단계: Step 2 — 청킹 핵심 로직 구현
""")
